# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas72O5/flyrank-ml-internship_week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = One unique content item (content_hash_id) for a specific client (client_hash_id).
Window: Features are derived from the mid-panel month of March 2026 (month=2026-03).

In [3]:
from datasets import load_dataset
from google.colab import userdata
import pandas as pd

# 1. Get your token from Colab Secrets
token = userdata.get('HF_TOKEN')

print("Connecting to FlyRank Warehouse...")

# 2. Load the dataset for March 2026
# This handles the gating, the token, and the file paths for you.
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/*.parquet",
    token=token,
    streaming=False # Set to True if the dataset is too large for your RAM
)

# Convert to Pandas
df_march = dataset['train'].to_pandas()

print(f"Success! Loaded {len(df_march):,} rows for March 2026.")

# --- VERIFICATION QUERIES ---

# Query 1: Prove the Grain (One row per page/client per day)
grain = df_march.groupby(['client_hash_id', 'content_hash_id']).size().reset_index(name='days_count')
print("\nFact 1: The Grain (Days per page in March)")
display(grain.head(5))

# Query 2: Row count and Date span
print(f"\nFact 2: Row Count: {len(df_march):,}")
print(f"Fact 2: Date Span: {df_march['report_date'].min()} to {df_march['report_date'].max()}")

# Query 3: Availability Check
gsc_ok = df_march['gsc_data_available'].sum()
ga4_ok = df_march['ga4_data_available'].sum()
print(f"\nFact 3: Rows with GSC data: {gsc_ok:,}")
print(f"Fact 3: Rows with GA4 data: {ga4_ok:,}")

Connecting to FlyRank Warehouse...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Success! Loaded 9,841,378 rows for March 2026.

Fact 1: The Grain (Days per page in March)


,client_hash_id,content_hash_id,days_count
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,31
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,31
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,31
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,31
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,31



Fact 2: Row Count: 9,841,378
Fact 2: Date Span: 2026-03-01 to 2026-03-31

Fact 3: Rows with GSC data: 3,611,061
Fact 3: Rows with GA4 data: 413,966


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Features (Clues): content_age_days, avg_position, ctr, sessions_90d, word_count. (Available at decision time).
Label (Target): is_decaying_champion. (A proxy created from trend_direction == 'down' where avg_position < 20).
Context: client_hash_id, content_hash_id. (Used for grouping and client-holdout).
Excluded: trend_pct and health_score.
Why: trend_pct is the exact number used to calculate the label (Leakage). health_score is a product-calculated decision; using it would be circular reasoning.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# CODE CELL for Section 2
# Exact names from your warehouse printout
feature_cols = [
    'gsc_impressions',
    'gsc_avg_position',
    'ga4_sessions',
    'ga4_engaged_sessions',
    'sessions_ai'
]

# Verify features exist
feature_verify = df_march[feature_cols].head(1)
print("Features confirmed available in warehouse:")
display(feature_verify)

Features confirmed available in warehouse:


,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,sessions_ai
0,20,3.35,NaN,NaN,NaN


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# CODE CELL for Section 3
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# 1. Row Counts: Unique pages in this slice
total_pages = df_march['content_hash_id'].nunique()

# 2. Availability Check: GSC vs GA4
# Note: Using the exact column names from your list
gsc_ok = df_march['gsc_data_available'].sum()
ga4_ok = df_march['ga4_data_available'].sum()

# 3. Create our Target Label (The 'Proxy')
# We define a low-performance page as one that has 0 clicks
y = (df_march['gsc_clicks'] == 0).astype(int)

print(f"Total Unique Pages: {total_pages:,}")
print(f"Rows with GSC Data: {gsc_ok:,}")
print(f"Rows with GA4 Data: {ga4_ok:,}")
print(f"Target Label ('Low Click' pages) prevalence: {y.mean():.2%}")

# --- THE TRAP: LEAKAGE EXPERIMENT ---
# We add 'gsc_clicks' to the features.
# Since we are trying to predict if clicks == 0, knowing the clicks is a LEAK.
X_leaky = df_march[feature_cols + ['gsc_clicks']].fillna(0)

model = RandomForestClassifier(max_depth=2, random_state=42).fit(X_leaky, y)
print(f"\n[TRAP] Leaky Model Score: {model.score(X_leaky, y):.4f} (Perfect 1.0 = Leakage!)")

# --- THE HONEST MODEL ---
X_honest = X_leaky.drop(columns=['gsc_clicks'])
model.fit(X_honest, y)
print(f"[HONEST] Honest Model Score: {model.score(X_honest, y):.4f}")

Total Unique Pages: 331,437
Rows with GSC Data: 3,611,061
Rows with GA4 Data: 413,966
Target Label ('Low Click' pages) prevalence: 95.75%

[TRAP] Leaky Model Score: 1.0000 (Perfect 1.0 = Leakage!)
[HONEST] Honest Model Score: 0.9586


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. Unbalanced History: As noted in the dim_clients table, some clients have 12+ months of history, while others have only 3. This may introduce bias toward more established websites.
2. Tracking Gaps: Rows from before a client's ga4_data_start contain search data only. I must use ga4_data_available IS TRUE as a filter if I want to use engagement metrics.
3. Observational Only: The data shows what happened, but not why. It cannot distinguish between a content problem and a sudden competitor bid increase.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Prove the 'Unbalanced History' limitation
# Check how many unique days of data each client has in the month
client_history = df_march.groupby('client_hash_id')['report_date'].nunique().reset_index()
client_history.columns = ['client_hash_id', 'days_of_history']

print("Evidence of Unbalanced history (Top 5 clients by data density in March):")
display(client_history.sort_values('days_of_history', ascending=False).head(5))

Evidence of Unbalanced history (Top 5 clients by data density in March):


,client_hash_id,days_of_history
0,client_0797ff3a1fc9a6a5,31
1,client_08a6a72ff48e62c0,31
2,client_08d2847f24cf89c1,31
3,client_0e1acc6cd57b0eba,31
4,client_0fa64a184f18a4a0,31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.